# 1. Vectors in Azure SQL

The embeddings live in a plain table column of type `vector(1536)`. No separate vector database: the search is a T-SQL query.

In [ ]:
import json
import time

from demo.notebook import embed_one, execute, query

## An embedding from Microsoft Foundry

`text-embedding-3-small` turns a question into 1,536 numbers. The question shares almost no words with the episode that answers it.

In [ ]:
question = "gdzie tanio postawić mały projekt po godzinach"
vector = embed_one(question)  # JSON text, the parameter of CAST(? AS vector(1536))
values = json.loads(vector)
print(len(values), values[:5])
assert len(values) == 1536, "text-embedding-3-small returns 1,536 dimensions"

## The `vector` column

One row per speaker turn; the embedding sits next to the text.

In [ ]:
turns = query("""
SELECT TOP (3)
    episode_number, chunk_number, speaker,
    VECTORPROPERTY(embedding, 'Dimensions') AS dimensions,
    VECTORPROPERTY(embedding, 'BaseType') AS base_type,
    LEFT(CAST(embedding AS NVARCHAR(MAX)), 60) AS embedding_as_text
FROM dbo.TranscriptChunks
ORDER BY chunk_id
""")
display(turns)
wrong = query("""
SELECT COUNT(*) AS wrong_dimensions
FROM dbo.TranscriptChunks
WHERE VECTORPROPERTY(embedding, 'Dimensions') <> 1536
""")
assert wrong.wrong_dimensions[0] == 0, "every embedding in the table has 1,536 dimensions"

## Exact search: `VECTOR_DISTANCE`

A full scan of all turns, sorted by cosine distance. Watch the top hits: episode 155 ("Bieda-hosting") without the words of the question.

In [ ]:
exact_sql = """
SELECT TOP (10)
    c.chunk_id, c.episode_number, c.speaker, c.transcript_text,
    VECTOR_DISTANCE('cosine', c.embedding, CAST(? AS vector(1536))) AS distance
FROM dbo.TranscriptChunks AS c
ORDER BY distance
"""
start = time.perf_counter()
exact = query(exact_sql, (vector,))
print(f"exact scan: {(time.perf_counter() - start) * 1000:.0f} ms")
display(exact)
assert len(exact) == 10, "the exact search returns 10 rows"
assert exact.episode_number[0] == 155, "the best turn comes from episode 155"

## A vector index (DiskANN)

Optional: **rebuild from scratch — takes about a minute on S0.** Set `rebuild = True` to drop the index and show the build live.

In [ ]:
rebuild = False
if rebuild:
    execute("DROP INDEX IF EXISTS IX_TranscriptChunks_embedding ON dbo.TranscriptChunks")

Create the index only when it is missing: instant when it was built before the talk.

In [ ]:
start = time.perf_counter()
built = query("""
IF NOT EXISTS (SELECT 1 FROM sys.indexes
               WHERE object_id = OBJECT_ID('dbo.TranscriptChunks')
                 AND name = 'IX_TranscriptChunks_embedding')
BEGIN
    CREATE VECTOR INDEX IX_TranscriptChunks_embedding
    ON dbo.TranscriptChunks (embedding)
    WITH (METRIC = 'cosine', TYPE = 'DiskANN');
    SELECT 1 AS built;
END
ELSE
    SELECT 0 AS built;
""").built[0]
print(f"index {'built' if built else 'already there'}: {time.perf_counter() - start:.1f} s")
index = query("""
SELECT name, type_desc
FROM sys.indexes
WHERE object_id = OBJECT_ID('dbo.TranscriptChunks') AND name = 'IX_TranscriptChunks_embedding'
""")
display(index)
assert len(index) == 1, "the vector index exists"

## Approximate search: `VECTOR_SEARCH`

`TOP (10) WITH APPROXIMATE` uses the index. Compare it with the exact top 10.

In [ ]:
approximate_sql = """
DECLARE @question vector(1536) = CAST(? AS vector(1536));
SELECT TOP (10) WITH APPROXIMATE
    c.chunk_id, c.episode_number, c.publication_date, c.speaker, c.transcript_text, s.distance
FROM VECTOR_SEARCH(
    TABLE = dbo.TranscriptChunks AS c, COLUMN = embedding,
    SIMILAR_TO = @question, METRIC = 'cosine') AS s
ORDER BY s.distance
"""
start = time.perf_counter()
approximate = query(approximate_sql, (vector,))
print(f"approximate search: {(time.perf_counter() - start) * 1000:.0f} ms")
print(f"overlap with the exact top 10: {len(set(exact.chunk_id) & set(approximate.chunk_id))}/10")
display(approximate)
assert len(approximate) == 10, "the approximate search returns 10 rows"

## Approximate search with a filter

The `WHERE` on `publication_date` applies during the index search, not after it.

In [ ]:
filtered_sql = """
DECLARE @question vector(1536) = CAST(? AS vector(1536));
SELECT TOP (10) WITH APPROXIMATE
    c.episode_number, c.publication_date, c.speaker, c.transcript_text, s.distance
FROM VECTOR_SEARCH(
    TABLE = dbo.TranscriptChunks AS c, COLUMN = embedding,
    SIMILAR_TO = @question, METRIC = 'cosine') AS s
WHERE c.publication_date BETWEEN ? AND ?
ORDER BY s.distance
"""
from_date, to_date = "2025-01-01", "2025-06-30"
filtered = query(filtered_sql, (vector, from_date, to_date))
display(filtered)
assert len(filtered) >= 1, "the filtered search returns rows"
assert filtered.publication_date.astype(str).between(from_date, to_date).all(), "all rows in the range"

At 1,996 rows the exact scan is as fast as the index. The index matters at hundreds of thousands of rows; the query shape stays the same.